In [3]:
import os
import numpy as np
import pandas as pd

# UPDATE THIS PATH to wherever your Motor-2 folder is
BASE = 'Motor-2'

def p(speed, folder, ch):
    return os.path.join(BASE, str(speed), folder,
                        f'Electric_Motor-2_{speed}_time-{folder}-ch{ch}.csv')

files = {
    ('healthy', 100): {'A': p(100, 'healthy 1', 1), 'B': p(100, 'healthy 1', 2), 'C': p(100, 'healthy 1', 3)},
    ('healthy',  75): {'A': p(75,  'healthy 1', 1), 'B': p(75,  'healthy 1', 2), 'C': p(75,  'healthy 1', 3)},
    ('healthy',  50): {'A': p(50,  'healthy 1', 1), 'B': p(50,  'healthy 1', 2), 'C': p(50,  'healthy 1', 3)},
    ('brb',     100): {'A': p(100, 'broken rotor bar', 1), 'B': p(100, 'broken rotor bar', 2), 'C': p(100, 'broken rotor bar', 3)},
    ('brb',      75): {'A': p(75,  'broken rotor bar', 1), 'B': p(75,  'broken rotor bar', 2), 'C': p(75,  'broken rotor bar', 3)},
    ('brb',      50): {'A': p(50,  'broken rotor bar', 1), 'B': p(50,  'broken rotor bar', 2), 'C': p(50,  'broken rotor bar', 3)},
    ('swf',     100): {'A': p(100, 'stator short 1', 1), 'B': p(100, 'stator short 1', 2), 'C': p(100, 'stator short 1', 3)},
    ('swf',      75): {'A': p(75,  'stator short 1', 1), 'B': p(75,  'stator short 1', 2), 'C': p(75,  'stator short 1', 3)},
    ('swf',      50): {'A': p(50,  'stator short 1', 1), 'B': p(50,  'stator short 1', 2), 'C': p(50,  'stator short 1', 3)},
}

EXPECTED_ROWS = 300_000
OUTPUT_FILE   = 'mcsa_master_dataset.parquet'

# Verify all 27 files exist
print("Checking files...")
all_ok = True
for key, phases in files.items():
    for ph, path in phases.items():
        exists = '✅' if os.path.exists(path) else '❌'
        if not os.path.exists(path):
            all_ok = False
        print(f'  {exists} {key} Phase {ph}: {os.path.basename(path)}')
print(f"\n{'All files found — ready to build.' if all_ok else 'Fix missing paths before continuing.'}")

Checking files...
  ✅ ('healthy', 100) Phase A: Electric_Motor-2_100_time-healthy 1-ch1.csv
  ✅ ('healthy', 100) Phase B: Electric_Motor-2_100_time-healthy 1-ch2.csv
  ✅ ('healthy', 100) Phase C: Electric_Motor-2_100_time-healthy 1-ch3.csv
  ✅ ('healthy', 75) Phase A: Electric_Motor-2_75_time-healthy 1-ch1.csv
  ✅ ('healthy', 75) Phase B: Electric_Motor-2_75_time-healthy 1-ch2.csv
  ✅ ('healthy', 75) Phase C: Electric_Motor-2_75_time-healthy 1-ch3.csv
  ✅ ('healthy', 50) Phase A: Electric_Motor-2_50_time-healthy 1-ch1.csv
  ✅ ('healthy', 50) Phase B: Electric_Motor-2_50_time-healthy 1-ch2.csv
  ✅ ('healthy', 50) Phase C: Electric_Motor-2_50_time-healthy 1-ch3.csv
  ✅ ('brb', 100) Phase A: Electric_Motor-2_100_time-broken rotor bar-ch1.csv
  ✅ ('brb', 100) Phase B: Electric_Motor-2_100_time-broken rotor bar-ch2.csv
  ✅ ('brb', 100) Phase C: Electric_Motor-2_100_time-broken rotor bar-ch3.csv
  ✅ ('brb', 75) Phase A: Electric_Motor-2_75_time-broken rotor bar-ch1.csv
  ✅ ('brb', 75) Phase 

In [4]:
def load_phase(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found:\n  {filepath}")
    df = pd.read_csv(filepath, header=0)
    if df.shape[0] != EXPECTED_ROWS:
        raise ValueError(f"Expected {EXPECTED_ROWS} rows but got {df.shape[0]} in:\n  {filepath}")
    if df.shape[1] < 6:
        raise ValueError(f"Expected at least 6 columns but got {df.shape[1]} in:\n  {filepath}")
    time         = df.iloc[:, 0].values
    measurements = df.iloc[:, 1:6].values
    return time, measurements

In [5]:
all_groups = []

for (condition, speed), phase_files in files.items():
    print(f"  Loading: {condition.upper()} @ {speed}% speed ...")
    time_A, meas_A = load_phase(phase_files['A'])
    time_B, meas_B = load_phase(phase_files['B'])
    time_C, meas_C = load_phase(phase_files['C'])

    group_df = pd.DataFrame({
        'time' : time_A,
        'A_m1' : meas_A[:, 0], 'B_m1': meas_B[:, 0], 'C_m1': meas_C[:, 0],
        'A_m2' : meas_A[:, 1], 'B_m2': meas_B[:, 1], 'C_m2': meas_C[:, 1],
        'A_m3' : meas_A[:, 2], 'B_m3': meas_B[:, 2], 'C_m3': meas_C[:, 2],
        'A_m4' : meas_A[:, 3], 'B_m4': meas_B[:, 3], 'C_m4': meas_C[:, 3],
        'A_m5' : meas_A[:, 4], 'B_m5': meas_B[:, 4], 'C_m5': meas_C[:, 4],
        'speed'    : speed,
        'condition': condition,
    })

    signal_cols = [c for c in group_df.columns if c not in ('speed', 'condition')]
    group_df[signal_cols] = group_df[signal_cols].astype('float32')
    all_groups.append(group_df)
    print(f"    Done — shape: {group_df.shape}")

print("\n  Concatenating all groups...")
df3 = pd.concat(all_groups, ignore_index=True)
print(f"  Master table shape: {df3.shape}")

  Loading: HEALTHY @ 100% speed ...
    Done — shape: (300000, 18)
  Loading: HEALTHY @ 75% speed ...
    Done — shape: (300000, 18)
  Loading: HEALTHY @ 50% speed ...
    Done — shape: (300000, 18)
  Loading: BRB @ 100% speed ...
    Done — shape: (300000, 18)
  Loading: BRB @ 75% speed ...
    Done — shape: (300000, 18)
  Loading: BRB @ 50% speed ...
    Done — shape: (300000, 18)
  Loading: SWF @ 100% speed ...
    Done — shape: (300000, 18)
  Loading: SWF @ 75% speed ...
    Done — shape: (300000, 18)
  Loading: SWF @ 50% speed ...
    Done — shape: (300000, 18)

  Concatenating all groups...
  Master table shape: (2700000, 18)


In [6]:
print(f"  Saving to: {OUTPUT_FILE}")
df3.to_parquet(OUTPUT_FILE, index=False)
size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)

print("\n" + "="*55)
print("  MCSA MASTER DATASET — SUMMARY")
print("="*55)
print(f"  Shape      : {df3.shape[0]:,} rows × {df3.shape[1]} columns")
print(f"  File size  : {size_mb:.1f} MB")
print(f"\n  Rows per condition:")
print(df3['condition'].value_counts().to_string())
print(f"\n  Rows per speed:")
print(df3['speed'].value_counts().sort_index().to_string())
print(f"\n  Rows per (condition, speed):")
print(df3.groupby(['condition', 'speed']).size().to_string())
print("="*55)
df3.head()

  Saving to: mcsa_master_dataset.parquet

  MCSA MASTER DATASET — SUMMARY
  Shape      : 2,700,000 rows × 18 columns
  File size  : 190.0 MB

  Rows per condition:
condition
healthy    900000
brb        900000
swf        900000

  Rows per speed:
speed
50     900000
75     900000
100    900000

  Rows per (condition, speed):
condition  speed
brb        50       300000
           75       300000
           100      300000
healthy    50       300000
           75       300000
           100      300000
swf        50       300000
           75       300000
           100      300000


,time,A_m1,B_m1,C_m1,A_m2,B_m2,C_m2,A_m3,B_m3,C_m3,A_m4,B_m4,C_m4,A_m5,B_m5,C_m5,speed,condition
0,0.00005,3.008112,22.242159,-24.287834,25.737957,-9.874649,-15.778249,18.146170,-24.400459,5.771639,-22.832848,-7.520176,30.482080,-28.979891,24.467091,5.640152,100,healthy
1,0.00010,2.451916,22.851290,-24.350483,25.655579,-9.530465,-16.185856,18.521677,-24.407204,5.298334,-22.014360,-8.193046,30.141418,-29.249149,24.527622,5.827123,100,healthy
2,0.00015,1.823028,23.583891,-24.382334,25.782387,-9.209298,-16.485453,19.160528,-23.988369,4.356100,-22.580967,-7.967415,30.581587,-29.046022,23.797901,6.286219,100,healthy
3,0.00020,1.305648,24.197058,-24.521708,25.505650,-8.242023,-17.209127,19.483103,-24.220560,4.210232,-21.292551,-8.737899,30.063391,-29.277792,23.951210,6.467302,100,healthy
4,0.00025,0.647593,24.997971,-24.620201,25.733488,-8.325099,-17.378656,20.150885,-23.920687,3.236333,-21.665260,-8.753317,30.315128,-29.282063,23.489681,6.787510,100,healthy
